# tSZ angular power spectrum: FLAMINGO map vs hmfast theory

We compute the full-sky tSZ (Compton-$y$) angular power spectrum of the FLAMINGO L2p8_m9 map and compare it to the halo-model prediction from **hmfast**.

**Map side.** We subtract the monopole, run `healpy.anafast`, and deconvolve the HEALPix pixel window. The Compton-$y$ field is dimensionless, so $C_\ell^{yy}$ is dimensionless.

**Theory side.** hmfast halo model with the **D3A** FLAMINGO fiducial cosmology, a **tSZ tracer** with the **Arnaud (2010) GNFW** pressure profile and hydrostatic-bias **$B=1.0$**. We sum the 1-halo and 2-halo terms.

We plot $D_\ell = \ell(\ell+1)C_\ell/2\pi$ versus $\ell$ for both.

In [1]:
import os

# Limit JAX GPU memory allocation to 10% for prefilling
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.10'

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

import jax.numpy as jnp

from flamingo import paths
from flamingo.maps import read_map
from flamingo.catalogue import D3A_COSMOLOGY
from hmfast.halos import HaloModel
from hmfast.halos.profiles import GNFWPressureProfile
from hmfast.tracers import tSZTracer

ymap = read_map(paths.HYDRO_MAP)
NSIDE = hp.npix2nside(ymap.size)
print('NSIDE', NSIDE, '| mean y (monopole)', float(ymap.mean()))


NSIDE 4096 | mean y (monopole) 1.5360727091771686e-06


## Map power spectrum

Subtract the monopole, then `anafast`. We divide by the pixel window function squared to undo the smoothing from finite pixel size. The map is a full-sky lightcone; we report the sky fraction and correct $C_\ell$ by $1/f_{\rm sky}$.

In [2]:
LMAX = 6000

f_sky = float(np.mean(ymap != 0.0))
ymap_nomono = ymap - ymap.mean()          # subtract the monopole

cl_map = hp.anafast(ymap_nomono, lmax=LMAX)
ell = np.arange(cl_map.size)
pw = hp.pixwin(NSIDE, lmax=LMAX)          # pixel window function
cl_map = cl_map / (pw**2 * f_sky)         # deconvolve pixel window, fsky correction

dl_map = ell * (ell + 1) / (2 * np.pi) * cl_map
print(f'f_sky = {f_sky:.3f}  | computed C_ell up to lmax={LMAX}')


def log_bin(ell, dl, lmin=10, lmax=LMAX, nbin=30):
    """Average D_ell in log-spaced ell bins (drops empty bins)."""
    edges = np.logspace(np.log10(lmin), np.log10(lmax), nbin + 1)
    idx = np.digitize(ell, edges) - 1
    lb, db = [], []
    for b in range(nbin):
        sel = (idx == b) & (ell >= lmin)
        if sel.any():
            lb.append(ell[sel].mean())
            db.append(dl[sel].mean())
    return np.array(lb), np.array(db)


ell_b, dl_map_b = log_bin(ell, dl_map)

f_sky = 1.000  | computed C_ell up to lmax=6000


## Theory power spectrum (hmfast)

D3A cosmology, tSZ tracer with the Arnaud A10 GNFW pressure profile and $B=1.0$. We integrate the Limber-projected 1-halo and 2-halo terms over mass ($10^{11}$ to $10^{15.5}\,M_\odot$) and redshift ($0<z<3$, matching the lightcone depth).

The redshift integral uses a grid **dense at low $z$**. The low-$\ell$ signal is dominated by rare nearby massive haloes, sharply peaked at low $z$; a coarse uniform $z$ grid under-samples that peak, and the trapezoidal rule overshoots, producing a spurious bump near $\ell\sim100$. The quadratically-spaced grid removes it and is converged to ~1%.

In [3]:
# Arnaud et al. (2010) GNFW pressure profiles: two variants compared below.
A10_B1 = dict(P0=8.403, c500=1.177, gamma=0.3081, alpha=1.0510, beta=5.4905)
A10_B134 = dict(P0=6.05, c500=1.77, gamma=0.31, alpha=1.48, beta=4.55)

hm = HaloModel(cosmology=D3A_COSMOLOGY)

ell_th = jnp.logspace(1.0, np.log10(LMAX), 40)
m = jnp.logspace(11.0, 15.5, 60)      # physical Msun
z = jnp.geomspace(0.001, 3.0, 60)


def tsz_dl(profile_params, B):
    """Total tSZ D_ell from hmfast (Tinker08 HMF, 1h + 2h)."""
    profile = GNFWPressureProfile(**profile_params, B=B)
    tracer = tSZTracer(profile=profile)
    cl1h = np.asarray(hm.cl_1h(tracer, tracer, l=ell_th, m=m, z=z))
    cl2h = np.asarray(hm.cl_2h(tracer, tracer, l=ell_th, m=m, z=z))
    pref = np.asarray(ell_th) * (np.asarray(ell_th) + 1) / (2 * np.pi)
    return pref * (cl1h + cl2h)


dl_th_B1 = tsz_dl(A10_B1, B=1.0)
dl_th_B134 = tsz_dl(A10_B134, B=1.0)

ell_th = np.asarray(ell_th)
print('A10, B=1.0    D_ell at ell~3000:', float(np.interp(3000, ell_th, dl_th_B1)))
print('A10, B=1.34   D_ell at ell~3000:', float(np.interp(3000, ell_th, dl_th_B134)))

# # --- Same tSZ tracer (B=1), but with the Mira-Titan HMF (native M200c) ---
# from hmfast.halos.massfunc import MTHaloMass

# profile_B1 = GNFWPressureProfile(**A10_B1, B=1.0)
# tracer_B1 = tSZTracer(profile=profile_B1)
# mt = MTHaloMass(); mt.prepare(D3A_COSMOLOGY)
# hm_mt = HaloModel(cosmology=D3A_COSMOLOGY, halo_mass_function=mt)
# z_mt = jnp.geomspace(0.001, 2.0, 60)
# m_mt = jnp.logspace(np.log10(5e13), 15.5, 50)   # Mira-Titan valid mass range

# pref = ell_th * (ell_th + 1) / (2 * np.pi)
# cl1h_mt = np.asarray(hm_mt.cl_1h(tracer_B1, tracer_B1, l=jnp.asarray(ell_th), m=m_mt, z=z_mt))
# cl2h_mt = np.asarray(hm_mt.cl_2h(tracer_B1, tracer_B1, l=jnp.asarray(ell_th), m=m_mt, z=z_mt))
# dl_th_mt = pref * (cl1h_mt + cl2h_mt)
# print('Mira-Titan (A10, B=1) D_ell at ell~3000:', float(np.interp(3000, ell_th, dl_th_mt)))

I0000 00:00:1782459859.832418 2507709 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782459859.832772 2507709 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1782459860.580689 2507709 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782459860.580915 2507709 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


A10, B=1.0    D_ell at ell~3000: 1.5918300729577205e-12
A10, B=1.34   D_ell at ell~3000: 1.562874709584679e-12


## Theory error bars (Gaussian + 1-halo trispectrum)

Diagonal covariance of the **theory** bandpowers (full sky, $f_{\rm sky}=1$):
$$
\mathrm{Var}(D_\ell) = \underbrace{\frac{2D_\ell^2}{(2\ell+1)\,\Delta\ell\,f_{\rm sky}}}_{\text{Gaussian (Knox)}} +
\underbrace{\frac{T_\ell}{(4\pi f_{\rm sky})}}_{\text{1-halo trispectrum}},
$$
where $T_\ell = T^{yy}_{\ell\ell}\,[ \ell(\ell+1)/2\pi]^2$ is the trispectrum diagonal
converted from $C_\ell$ to $D_\ell$, with intrinsic scatter $\sigma_{\ln Y}=0.173$
($T\to T\,e^{8\sigma_{\ln Y}^2}$ on the full sky). On the full sky the trispectrum
term is much larger than the Gaussian term.

In [4]:
SIGMA_LNY = 0.173
scatter_boost = np.exp(8.0 * SIGMA_LNY**2)

profile_B1 = GNFWPressureProfile(**A10_B1, B=1.0)
tracer_B1 = tSZTracer(profile=profile_B1)

ell_np = np.asarray(ell_th)
m_np = np.asarray(m)
z_np = np.asarray(z)
pref_th = ell_np * (ell_np + 1.0) / (2.0 * np.pi)

# Knox Gaussian: effective Delta ell along the log-spaced theory grid
delta_ell = np.gradient(ell_np, np.log(ell_np))
sigma_g = np.sqrt(2.0 * dl_th_B1**2 / ((2.0 * ell_np + 1.0) * delta_ell * f_sky))

# 1-halo trispectrum (C_l) -> D_l diagonal, same sigma_lnY boost as compute_arnaudB1_full_cov.py
T_C = np.asarray(hm.trispectrum_1h(
    tracer_B1, None,
    jnp.asarray(ell_th), jnp.asarray(ell_th),
    jnp.asarray(m), jnp.asarray(z),
)) * scatter_boost
T_D_diag = np.diag(T_C) * pref_th**2
sigma_t = np.sqrt(T_D_diag / (4.0 * np.pi * f_sky))
sigma_tot = np.sqrt(sigma_g**2 + sigma_t**2)

i3000 = int(np.argmin(np.abs(ell_np - 3000)))
print(f"full sky @ ell~3000:  sigma_G = {sigma_g[i3000]:.3e}  "
      f"sigma_T = {sigma_t[i3000]:.3e}  T/G = {sigma_t[i3000]/sigma_g[i3000]:.1f}")

full sky @ ell~3000:  sigma_G = 5.125e-16  sigma_T = 1.151e-14  T/G = 22.5


## Low-$\ell$ (point-source) 1-halo normalization

At $\ell\theta_{\rm max}\ll1$ the flat-sky transform $\tilde y_\ell=2\pi\int\theta\,d\theta\,y(\theta)J_0(\ell\theta)\to Y$, the integrated Compton-$y$ flux, so with $\theta_{\rm max}=5\theta_{500c}$ each halo contributes $\tilde y_\ell\simeq Y_{5R500c}$. The 1-halo spectrum then becomes **white** (independent of $\ell$),
$$C_\ell^{yy,1h}\to\int dz\,\frac{dV}{dz\,d\Omega}\int dM\,\frac{dn}{dM}\,Y_{5R500c}^2(M,z),$$
which appears as $D_\ell\propto\ell(\ell+1)\approx\ell^2$. We read $Y_{5R500c}$ off the catalogue ($Y_{\rm ang}=Y_{5R500c,\rm Mpc^2}/d_A^2$) and evaluate this two ways: the direct full-sky shot noise $\frac1{4\pi}\sum_a Y_{\rm ang,a}^2$, and the Tinker08-HMF integral using the catalogue $\langle Y_{\rm ang}^2\rangle(M,z)$.

In [5]:
# --- Low-ell (point-source) 1-halo normalization from catalogue Y_5R500c ---
# For l*theta_max << 1 the harmonic transform y_ell -> Y = integrated Compton-y,
# so C_l^{1h} -> int dz (dV/dz/dOmega) int dM (dn/dM) Y_ang^2  (white in C_l,
# hence D_l ~ l^2). Y_ang = Y_5R500c_Mpc2 / d_A^2 is the angular integrated y [sr].
from flamingo.catalogue import load_catalogue
from hmfast.halos import MassDefinition
from hmfast.halos.massfunc import T08HaloMass

cat = load_catalogue(paths.HYDRO_CATALOGUE)
_z = cat['z'].values
_dA = cat['r_comoving_Mpc'].values / (1 + _z)             # proper Mpc (matches D3A)
y_ang = cat['Y_5R500c_Mpc2'].values / _dA**2             # integrated y over solid angle [sr]

# (a) Direct catalogue shot noise on the full sky: C = (1/4pi) sum_a Y_ang,a^2.
C_white_cat = np.sum(y_ang**2) / (4 * np.pi)

# (b) Tinker08-HMF version: catalogue <Y_ang^2> in (log M500c, z) bins, reweighted
#     by the T08 expected counts (replaces the single-realisation dn/dM by theory).
mE = np.arange(13.7, 15.7, 0.1)
zE = np.linspace(0.0, 3.0, 31)
sumy2, _, _ = np.histogram2d(np.log10(cat['M_500c_Msun'].values), _z, bins=[mE, zE], weights=y_ang**2)
cnt, _, _ = np.histogram2d(np.log10(cat['M_500c_Msun'].values), _z, bins=[mE, zE])
meanY2 = np.where(cnt > 0, sumy2 / np.maximum(cnt, 1), 0.0)

hm5 = HaloModel(cosmology=D3A_COSMOLOGY, mass_definition=MassDefinition(500, 'critical'),
                halo_mass_function=T08HaloMass())
mc = 0.5 * (mE[:-1] + mE[1:]); zc = 0.5 * (zE[:-1] + zE[1:])
dndlnM_w = np.asarray(hm5.halo_mass_function.halo_mass_function(hm5, jnp.asarray(10**mc), jnp.asarray(zc)))
dV_w = np.asarray(D3A_COSMOLOGY.comoving_volume_element(jnp.asarray(zc)))   # Mpc^3/sr
N_T08 = 4 * np.pi * dV_w[None, :] * dndlnM_w * (np.log(10) * 0.1) * (zE[1] - zE[0])  # counts per bin
C_white_T08 = np.sum(meanY2 * N_T08) / (4 * np.pi)

# White in C_l -> ~l^2 in D_l; valid for l*theta_max < 1, so show it for l <~ 300.
ell_w = np.logspace(1.0, np.log10(300.0), 40)
dl_white_cat = ell_w * (ell_w + 1) / (2 * np.pi) * C_white_cat
dl_white_T08 = ell_w * (ell_w + 1) / (2 * np.pi) * C_white_T08
print('C_white: catalogue=%.3e  Tinker08=%.3e' % (C_white_cat, C_white_T08))

C_white: catalogue=2.936e-16  Tinker08=2.421e-16


## $D_\ell$ vs $\ell$

In [6]:
fig, ax = plt.subplots(figsize=(7.2, 5.4))
ax.loglog(ell_b, dl_map_b, 'o', ms=4, color='crimson', label='FLAMINGO map (anafast)')
ax.loglog(ell_th, dl_th_B1, 'k-', lw=2,
          label=r'GNFW ($c_{500}$=1.177, $\gamma$=0.308, $B$=1.0)')
ax.fill_between(ell_th, dl_th_B1 - sigma_g, dl_th_B1 + sigma_g,
                color='k', alpha=0.18, label='theory ±1σ Gaussian')
ax.fill_between(ell_th, dl_th_B1 - sigma_tot, dl_th_B1 + sigma_tot,
                color='k', alpha=0.10, label='theory ±1σ Gaussian + trispectrum')
ax.loglog(ell_th, dl_th_B134, 'C1-', lw=2,
          label=r'GNFW ($c_{500}$=0.869, $\gamma$=0.208, $B$=1.34)')
ax.set_xlabel(r'Multipole $\ell$')
ax.set_ylabel(r'$D_\ell = \ell(\ell+1)C_\ell^{yy}/2\pi$')
ax.set_title('tSZ angular power spectrum: FLAMINGO map vs A10 GNFW (D3A)')
ax.set_xlim(10, LMAX)
ax.legend(fontsize=8)
fig.tight_layout(); plt.show()